In [1]:
%pip install langchain>=1.0 | tail -n 1 
%pip install langchain-ibm==1.1.1 | tail -n 1 
%pip install langchain-community==0.4.2 | tail -n 1 
%pip install wikipedia==1.4.0 | tail -n 1
%pip install openai==1.77.0 | tail -n 1
%pip install langchain-openai==1.6.6 | tail -n 1

Note: you may need to restart the kernel to use updated packages.
  DEPRECATION: Building 'ibm-cos-sdk' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ibm-cos-sdk'. Discussion can be found at https://github.com/pypa/pip/issues/6334

  DEPRECATION: Building 'ibm-cos-sdk-core' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ibm-cos-sdk-core'. Discussion can be found at https://github.com/pypa/pip/iss

In [1]:
from langchain_ibm import ChatWatsonx
from ibm_watsonx_ai.wml_resource import WatsonxAPIWarning
from langchain.agents import create_agent

import re
import warnings
warnings.filterwarnings("ignore", category=WatsonxAPIWarning)

In [2]:
llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
)

In [3]:
response = llm.invoke("What is tool calling in langchain?")
print("\nResponse Content: ", response.content)


Response Content:  Tool calling in Langchain refers to a feature that allows a language model to interact with external tools or APIs to carry out specific tasks. Langchain is a Python library designed to assist with natural language processing (NLP) tasks and to facilitate the development of applications that require understanding or generating human language content.

Here’s a more detailed breakdown of tool calling in Langchain:

1. **Purpose**: The primary purpose of tool calling is to enable a language model to make use of pre-existing functionalities (tools) or APIs that can perform specific tasks which the model itself might not be able to do, such as querying a database, making an HTTP request, or accessing a particular dataset.

2. **Integration with Language Models**: Langchain integrates with various language models, allowing these models to call external tools in a seamless way. This is particularly useful in scenarios where a language model needs to provide accurate, up-t

In [4]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

In [5]:
add_numbers("1 2")

{'result': 3}

In [6]:
from langchain_core.tools import Tool
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)

tool object name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x7e936afdf880>

In [7]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x7e936afdf880>)>

In [8]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

Calling Tool Function:
{'result': 60}

In [9]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs: str) -> dict:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return {"result": sum(numbers)}

In [10]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 


Name: 
 add_numbers
Description: 
 Adds all numeric values found in the input string.
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}

In [11]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))  # Example

{'result': 60.0}

In [12]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}

In [13]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [14]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Args Schema Info: {'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}

In [15]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2

In [16]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [17]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return sum(numbers)

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_tool]
)

In [19]:
# Use the agent
response = agent.invoke({
    "messages": [("human", "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")]
})
print(response["messages"][-1].content)

The combined GDP of the United States, Canada, and Mexico in 2023 was **approximately $31.65 trillion**.

In [20]:
response

{'messages': [HumanMessage(content="In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.", additional_kwargs={}, response_metadata={}, id='dcf96b3b-88e5-48b3-a43a-d20307a74c77'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 235, 'total_tokens': 276}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-29ffd978740619b3594b39a6a24c752e', tool_calls=[{'name': 'AddTool', 'args': {'__arg1': '27.72 + 2.14 + 1.79'}, 'id': 'chatcmpl-tool-987b29ce3c631fb7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 235, 'output_tokens': 41, 'total_tokens': 276}),
  ToolMessage(content='{"result": 0}', name='AddTool', id='c21126b7-a6f6-4672-9ea5-0db973da3f54', tool_call_id='chatcmpl-tool-987b29ce3c631fb7'),
  AIMessage(content='', additional_kwa

In [21]:
agent.invoke({"messages": [("human", "Add 10, 20, two and 30")]})

{'messages': [HumanMessage(content='Add 10, 20, two and 30', additional_kwargs={}, response_metadata={}, id='120bbd35-d5c4-4f32-9589-76cbc960137d'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 205, 'total_tokens': 243}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-f5e949faefc8cb473ce6545c9b992549', tool_calls=[{'name': 'AddTool', 'args': {'__arg1': '10, 20, 2, 30'}, 'id': 'chatcmpl-tool-a17b8fae52e6ffe7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 38, 'total_tokens': 243}),
  ToolMessage(content='{"result": 62}', name='AddTool', id='ea33c177-0896-4593-b64b-0e86c8d0388d', tool_call_id='chatcmpl-tool-a17b8fae52e6ffe7'),
  AIMessage(content='The sum of 10, 20, 2, and 30 is **62**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 255, 'total

In [22]:
agent_2 = create_agent(
    model=llm,
    tools=[sum_numbers_from_text]
)
response = agent_2.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response["messages"][-1].content)

The sum of the absolute values of -10, -20, and -30 is **60**.

In [23]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-5.4-nano")

In [24]:
agent_3 = create_agent(
    model=llm_ai,
    tools=[sum_numbers_with_complex_output]
)
response = agent_3.invoke({"messages": [("human", "Add 10, 20 and 30")]})
print(response["messages"][-1].content)

60

In [25]:
agent_2 = create_agent(
    model=llm,
    tools=[add_numbers_with_options]
)

In [26]:
response = agent_2.invoke({
    "messages": [("Add -10, -20, and -30 using absolute values.")]
})
print(response)

{'messages': [HumanMessage(content='Add -10, -20, and -30 using absolute values.', additional_kwargs={}, response_metadata={}, id='f5b87f19-05c9-4602-bb38-b4cd361f151c'), AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 275, 'total_tokens': 317}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-a29095852c0bceb8cb904402aa02a47b', tool_calls=[{'name': 'add_numbers_with_options', 'args': {'numbers': [-10, -20, -30], 'absolute': True}, 'id': 'chatcmpl-tool-8ab603b9a4180c1a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 275, 'output_tokens': 42, 'total_tokens': 317}), ToolMessage(content='60.0', name='add_numbers_with_options', id='ce795155-8a30-476e-a57d-2324f9af5bf7', tool_call_id='chatcmpl-tool-8ab603b9a4180c1a'), AIMessage(content='The sum of the absolute values of -10, -20, and -30 is **60**.', additional_kwargs={}, respon

In [27]:
agent_openai = create_agent(
    model=llm_ai,
    tools=[add_numbers_with_options]
)

In [28]:
response = agent_openai.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response)

{'messages': [HumanMessage(content='Add -10, -20, and -30 using absolute values.', additional_kwargs={}, response_metadata={}, id='2020ddbe-3b9d-43df-9e22-60d31cd857a9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 201, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-ETJBsNvk1ajxwunTZK9vCxjFTmXYD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0eb4b-786c-7bf3-b5e1-fd1abd5b346a-0', tool_calls=[{'name': 'add_numbers_with_options', 'args': {'numbers': [-10, -20, -30], 'absolut

In [29]:
%pip install langgraph==0.6.1 | tail -n 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.4.3 requires langgraph<1.3.0,>=1.2.11, but you have langgraph 0.6.1 which is incompatible.

Note: you may need to restart the kernel to use updated packages.

In [30]:
from langchain.agents import create_agent

agent_exec = create_agent(model=llm_ai, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [31]:
print(msgs["messages"][-1].content)

The sum of -10, -20, and -30 is **-60**.

In [32]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and successively subtracts each number after the first
    from the first number.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": 70}
    """
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    result = numbers[0]
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [33]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Name: 
 subtract_numbers
Description: 
 Extracts numbers from a string and successively subtracts each number after the first
from the first number.

Example Input:
"100, 20, 10"

Example Output:
{"result": 70}
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}

In [34]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(subtract_numbers.invoke(test_input))  # Example

Calling Tool Function:
{'result': -40.0}

In [35]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and calculates their product."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 1}

    result = 1
    for num in numbers:
        result *= num
    return {"result": result}

In [36]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and divides the first by each subsequent number."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    if any(num == 0 for num in numbers[1:]):
        return {"result": "Cannot divide by zero."}

    result = numbers[0]
    for num in numbers[1:]:
        result /= num
    return {"result": result}

In [37]:
# Testing multiply_tool
multiply_test_input = "2, 3, and 4"
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

--- Testing MultiplyTool ---
Input: 2, 3, and 4
Output: {'result': 24.0}

In [38]:
# Testing divide_tool
divide_test_input = "100, 5, 2"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

--- Testing DivideTool ---
Input: 100, 5, 2
Output: {'result': 10.0}

In [39]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds all numeric values found in the input string.', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x7e9369988360>),
 StructuredTool(name='subtract_numbers', description='Extracts numbers from a string and successively subtracts each number after the first\nfrom the first number.\n\nExample Input:\n"100, 20, 10"\n\nExample Output:\n{"result": 70}', args_schema=<class 'langchain_core.utils.pydantic.subtract_numbers'>, func=<function subtract_numbers at 0x7e93684b2700>),
 StructuredTool(name='multiply_numbers', description='Extracts numbers from a string and calculates their product.', args_schema=<class 'langchain_core.utils.pydantic.multiply_numbers'>, func=<function multiply_numbers at 0x7e93682b13a0>),
 StructuredTool(name='divide_numbers', description='Extracts numbers from a string and divides the first by each subsequent number.', args_schema=<class 'langchain_core.utils.pydantic.divi

In [40]:
# Create the agent with all tools
math_agent = create_agent(
    model=llm_ai,
    tools=tools,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [41]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

25 ÷ 4 = **6.25**.

In [42]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

{"result": 70.0}

In [43]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and 4.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)


--- Testing MultiplyTool ---
Agent Response: \(2 \times 3 \times 4 = 24\).

--- Testing DivideTool ---
Agent Response: \(100 \div 5 = 20\), and then \(20 \div 2 = 10\).

In [44]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [45]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_agent(
    model=llm,
    tools=tools_updated,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent", math_agent_new)

agent <langgraph.graph.state.CompiledStateGraph object at 0x7e93682b64b0>

In [46]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [47]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.
Tool Result: 70
Expected Result: 70
✅ Test Passed: Testing subtraction tool with sequential subtraction.

--- Test Case 2: Testing multiplication tool for a list of numbers. ---
Query: Multiply 2, 3, and 4.
Tool Result: 24.0
Expected Result: 24
✅ Test Passed: Testing multiplication tool for a list of numbers.

--- Test Case 3: Testing division tool with sequential division. ---
Query: Divide 100 by 5 and then by 2.
Tool Result: 10.0
Expected Result: 10.0
✅ Test Passed: Testing division tool with sequential division.

--- Test Case 4: Testing subtraction tool with negative results. ---
Query: Subtract 50 from 20.
Tool Result: -30
Expected Result: -30
✅ Test Passed: Testing subtraction tool with negative results.

Correctly passed tests: ['Testing subtraction tool with sequential subtraction.', 'Testing multiplication tool for a list of numbers.', 'Testing division tool with seque

In [48]:
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia.wikipedia as wp
import requests as _requests
import time

wp.API_URL = "https://en.wikipedia.org/w/api.php"

def _patched_request(params, _retries=3):
    params['format'] = 'json'
    if 'action' not in params:
        params['action'] = 'query'
    headers = {"User-Agent": "AI-Math-Assistant-Lab/1.0 (contact@example.com)"}

    last_exc = None
    for attempt in range(_retries):
        r = _requests.get(wp.API_URL, params=params, headers=headers)
        try:
            return r.json()
        except ValueError as e:
            last_exc = e
            print(f"Attempt {attempt+1}/{_retries} failed — status {r.status_code}, body: {r.text[:200]!r}")
            time.sleep(1)
    raise last_exc

wp._wiki_request = _patched_request

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

/tmp/ipykernel_416/1445578807.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper

In [49]:
search_wikipedia.invoke("What is tool calling?")

'Page: Veronika (cow)\nSummary: Veronika (born 2012 or 2013) is a pet Braunvieh cow in the Austrian town of Nötsch im Gailtal. She became known in January 2026 after being observed using tools to scratch herself. The incident has been described as the first documented case of cattle using tools, and has led scientists to reassess the intelligence of cows.\n\nPage: Cold calling\nSummary: Cold calling is the solicitation of business from potential customers who have had no prior contact with the salesperson conducting the call. It is an attempt to convince potential customers to purchase the salesperson\'s product or service.  Generally, it is an over-the-phone process, making it a form of telemarketing, but can also be done in-person by door-to-door salespeople.  Though cold calling is common as a legitimate business technique, scammers can use cold calling as well.\nCold calling allows businesses to reach new customers at low cost, and has a particular advantage of tending to involve k

In [50]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_agent(
    model=llm_ai,
    tools=tools_updated,
    system_prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")
    print ('hello world')


Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'population of Canada'}, 'id': 'call_jHOkWwsA0AO8yfyyJ8X3uK2B', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Population of Canada
Summary: Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of Q1 2026. Despite being the second-largest country by total area (fourth-largest by land area), the vast majority of the country is sparsely inhabited, with most of its population south of the 55th parallel north. Just over 60% of Canadians live in just two provinces: Ontario and Quebec. Though Canada's overall population density is low, many regions in the south, such as the Quebec City–Windsor Corridor, have population densities highe